<a href="https://colab.research.google.com/github/mhusef/MCS_AI/blob/A2/A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import os
import random
import string
import warnings
warnings.filterwarnings('ignore')

# Function to create random folder name in A2 directory
def create_run_folder():
    random_name = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    # Create folder inside A2 directory
    # Use os.getcwd() instead of __file__ as __file__ is not defined in Colab
    base_dir = os.getcwd()
    folder_name = os.path.join(base_dir, f"run_{random_name}")

    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        # Create subfolders for different datasets
        os.makedirs(os.path.join(folder_name, 'diabetes'))
        os.makedirs(os.path.join(folder_name, 'skin_segmentation'))

    return folder_name

# class to load and preprocess datasets
class DataProcessor:
    def __init__(self):
        self.scaler = StandardScaler()

    # load Pima Indians Diabetes dataset
    def load_diabetes_data(self):
        # Use os.getcwd() instead of __file__ as __file__ is not defined in Colab
        base_dir = os.getcwd()
        csv_path = os.path.join(base_dir, 'diabetes.csv')
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Error: diabetes.csv not found at {csv_path}")
        df = pd.read_csv(csv_path)

        # Convert 'Outcome' column to 'target'
        if 'Outcome' in df.columns:
            df['target'] = df['Outcome']
            df = df.drop('Outcome', axis=1)

        print(f"Loaded diabetes dataset: {len(df)} samples, {len(df.columns)-1} features")
        return df

    # load  Skin Segmentation dataset
    def load_skin_data(self):
        # Use os.getcwd() instead of __file__ as __file__ is not defined in Colab
        base_dir = os.getcwd()
        txt_path = os.path.join(base_dir, 'Skin_NonSkin.txt')
        if not os.path.exists(txt_path):
            raise FileNotFoundError(f"Error: Skin_NonSkin.txt not found at {txt_path}")
        df = pd.read_csv(txt_path, sep='\t', header=None, names=['B', 'G', 'R', 'Class'])
        # Convert Class: 1=skin, 2=non-skin -> 1=skin, 0=non-skin
        df['target'] = (df['Class'] == 1).astype(int)
        df = df.drop('Class', axis=1)
        print(f"Loaded skin segmentation dataset: {len(df)} samples, {len(df.columns)-1} features")
        return df

    # preprocess diabetes data
    def preprocess_diabetes_data(self, df):
        # select all features except the target column
        feature_cols = [col for col in df.columns if col != 'target']
        X = df[feature_cols]
        y = df['target']
        # split the data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        # fit on training data only, then transform both sets
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols

    # preprocess skin segmentation data
    def preprocess_skin_data(self, df):
        # select all features except the target column
        feature_cols = [col for col in df.columns if col != 'target']
        X = df[feature_cols]
        y = df['target']
        # split the data into training and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        # fit on training data only, then transform both sets
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols

# class for linear SVM analysis
class LinearSVMAnalyzer:
    def __init__(self, output_folder=".", dataset_name="dataset"):
        self.model = SVC(kernel='linear', random_state=42)
        self.output_folder = os.path.join(output_folder, dataset_name)
        self.dataset_name = dataset_name

    def train_model(self, X_train, X_test, y_train, y_test):
        # Train the linear SVM
        self.model.fit(X_train, y_train)

        # Make predictions
        y_train_pred = self.model.predict(X_train)
        y_test_pred = self.model.predict(X_test)

        # Calculate accuracies
        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)

        # Get confusion matrix and classification report
        conf_matrix = confusion_matrix(y_test, y_test_pred)
        class_report = classification_report(y_test, y_test_pred)

        return train_accuracy, test_accuracy, conf_matrix, class_report

    def plot_confusion_matrix(self, conf_matrix):
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Class 0', 'Class 1'],
                   yticklabels=['Class 0', 'Class 1'])
        plt.title(f'Linear SVM Confusion Matrix - {self.dataset_name.title()}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')

        output_path = os.path.join(self.output_folder, f'linear_svm_confusion_matrix.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

    def save_results(self, train_acc, test_acc, conf_matrix, class_report):
        results_path = os.path.join(self.output_folder, 'linear_svm_results.txt')
        with open(results_path, 'w') as f:
            f.write(f"LINEAR SVM RESULTS - {self.dataset_name.upper()}\n")
            f.write("="*50 + "\n\n")

            f.write("ACCURACY SCORES:\n")
            f.write(f"Training Accuracy: {train_acc:.4f}\n")
            f.write(f"Test Accuracy: {test_acc:.4f}\n\n")

            f.write("CONFUSION MATRIX:\n")
            f.write(f"{conf_matrix}\n\n")

            f.write("CLASSIFICATION REPORT:\n")
            f.write(f"{class_report}\n")

# class for non-linear SVM analysis
class NonLinearSVMAnalyzer:
    def __init__(self, output_folder=".", dataset_name="dataset"):
        self.kernels = ['rbf', 'poly']
        self.output_folder = os.path.join(output_folder, dataset_name)
        self.dataset_name = dataset_name
        self.models = {}
        self.results = {}

    def train_models(self, X_train, X_test, y_train, y_test):
        # Train RBF SVM
        rbf_model = SVC(kernel='rbf', random_state=42)
        rbf_model.fit(X_train, y_train)

        rbf_train_pred = rbf_model.predict(X_train)
        rbf_test_pred = rbf_model.predict(X_test)

        rbf_train_acc = accuracy_score(y_train, rbf_train_pred)
        rbf_test_acc = accuracy_score(y_test, rbf_test_pred)
        rbf_conf_matrix = confusion_matrix(y_test, rbf_test_pred)
        rbf_class_report = classification_report(y_test, rbf_test_pred)

        # Train Polynomial SVM
        poly_model = SVC(kernel='poly', degree=3, random_state=42)
        poly_model.fit(X_train, y_train)

        poly_train_pred = poly_model.predict(X_train)
        poly_test_pred = poly_model.predict(X_test)

        poly_train_acc = accuracy_score(y_train, poly_train_pred)
        poly_test_acc = accuracy_score(y_test, poly_test_pred)
        poly_conf_matrix = confusion_matrix(y_test, poly_test_pred)
        poly_class_report = classification_report(y_test, poly_test_pred)

        # Store models and results
        self.models['rbf'] = rbf_model
        self.models['poly'] = poly_model

        self.results['rbf'] = {
            'train_acc': rbf_train_acc,
            'test_acc': rbf_test_acc,
            'conf_matrix': rbf_conf_matrix,
            'class_report': rbf_class_report
        }

        self.results['poly'] = {
            'train_acc': poly_train_acc,
            'test_acc': poly_test_acc,
            'conf_matrix': poly_conf_matrix,
            'class_report': poly_class_report
        }

        return self.results

    def perform_cross_validation(self, X_train, y_train):
        cv_results = {}

        for kernel in self.kernels:
            if kernel == 'rbf':
                model = SVC(kernel='rbf', random_state=42)
            else:
                model = SVC(kernel='poly', degree=3, random_state=42)

            # Perform 5-fold cross validation for good balance of speed and reliability
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
            cv_results[kernel] = {
                'mean_score': cv_scores.mean(),
                'std_score': cv_scores.std(),
                'scores': cv_scores
            }

        return cv_results

    def tune_hyperparameters(self, X_train, y_train):
        # Optimized hyperparameter tuning for RBF kernel (faster but still comprehensive)
        rbf_params = {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
        }

        rbf_grid = GridSearchCV(
            SVC(kernel='rbf', random_state=42),
            rbf_params,
            cv=3,
            scoring='accuracy',
            n_jobs=-1,
            verbose=0
        )
        rbf_grid.fit(X_train, y_train)

        # Optimized hyperparameter tuning for Polynomial kernel (reduced search space)
        poly_params = {
            'C': [0.1, 1, 10, 100],
            'degree': [2, 3, 4],
            'gamma': ['scale', 'auto', 0.01, 0.1],
            'coef0': [0, 0.1, 1]
        }

        poly_grid = GridSearchCV(
            SVC(kernel='poly', random_state=42),
            poly_params,
            cv=3,
            scoring='accuracy',
            n_jobs=-1,
            verbose=0
        )
        poly_grid.fit(X_train, y_train)

        return {
            'rbf': {'best_params': rbf_grid.best_params_, 'best_score': rbf_grid.best_score_},
            'poly': {'best_params': poly_grid.best_params_, 'best_score': poly_grid.best_score_}
        }

    def plot_kernel_comparison(self):
        kernels = list(self.results.keys())
        test_accuracies = [self.results[k]['test_acc'] for k in kernels]

        plt.figure(figsize=(10, 6))
        bars = plt.bar(kernels, test_accuracies, color=['skyblue', 'lightcoral'])
        plt.title(f'SVM Kernel Comparison - {self.dataset_name.title()}')
        plt.ylabel('Test Accuracy')
        plt.xlabel('Kernel Type')
        plt.ylim(0, 1)

        # Add value labels on bars
        for bar, acc in zip(bars, test_accuracies):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{acc:.3f}', ha='center', va='bottom')

        plt.grid(True, alpha=0.3)
        output_path = os.path.join(self.output_folder, 'kernel_comparison.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

    def save_results(self, cv_results, tuning_results):
        results_path = os.path.join(self.output_folder, 'nonlinear_svm_results.txt')
        with open(results_path, 'w') as f:
            f.write(f"NON-LINEAR SVM RESULTS - {self.dataset_name.upper()}\n")
            f.write("="*60 + "\n\n")

            # Write kernel comparison
            f.write("KERNEL PERFORMANCE COMPARISON:\n")
            f.write("-" * 40 + "\n")
            for kernel in self.kernels:
                result = self.results[kernel]
                f.write(f"{kernel.upper()} Kernel:\n")
                f.write(f"  Training Accuracy: {result['train_acc']:.4f}\n")
                f.write(f"  Test Accuracy: {result['test_acc']:.4f}\n\n")

            # Write cross-validation results
            f.write("CROSS-VALIDATION RESULTS:\n")
            f.write("-" * 40 + "\n")
            for kernel, cv_result in cv_results.items():
                f.write(f"{kernel.upper()} Kernel CV:\n")
                f.write(f"  Mean Score: {cv_result['mean_score']:.4f} (+/- {cv_result['std_score']*2:.4f})\n")
                f.write(f"  Individual Scores: {cv_result['scores']}\n\n")

            # Write hyperparameter tuning results
            f.write("HYPERPARAMETER TUNING RESULTS:\n")
            f.write("-" * 40 + "\n")
            for kernel, tuning_result in tuning_results.items():
                f.write(f"{kernel.upper()} Kernel Best Parameters:\n")
                f.write(f"  Parameters: {tuning_result['best_params']}\n")
                f.write(f"  Best CV Score: {tuning_result['best_score']:.4f}\n\n")

            # Find best overall model
            best_kernel = max(self.results.keys(), key=lambda k: self.results[k]['test_acc'])
            f.write(f"BEST PERFORMING MODEL: {best_kernel.upper()} kernel\n")
            f.write(f"Test Accuracy: {self.results[best_kernel]['test_acc']:.4f}\n")

# class to compare all models
class SVMComparison:
    def __init__(self, output_folder=".", dataset_name="dataset"):
        self.output_folder = os.path.join(output_folder, dataset_name)
        self.dataset_name = dataset_name

    def compare_all_models(self, linear_results, nonlinear_results):
        # Combine all results
        all_results = {
            'Linear': linear_results[1],  # test accuracy
            'RBF': nonlinear_results['rbf']['test_acc'],
            'Polynomial': nonlinear_results['poly']['test_acc']
        }

        # Create comparison plot
        plt.figure(figsize=(12, 8))

        models = list(all_results.keys())
        accuracies = list(all_results.values())

        bars = plt.bar(models, accuracies, color=['lightblue', 'lightgreen', 'lightcoral'])
        plt.title(f'SVM Model Comparison - {self.dataset_name.title()}')
        plt.ylabel('Test Accuracy')
        plt.xlabel('SVM Model Type')
        plt.ylim(0, 1)

        # Add value labels on bars
        for bar, acc in zip(bars, accuracies):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

        plt.grid(True, alpha=0.3)
        output_path = os.path.join(self.output_folder, 'svm_model_comparison.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

        return all_results

def main():
    # Create a new folder for this run
    run_folder = create_run_folder()

    data_processor = DataProcessor()

    print("Loading and preprocessing datasets...")

    # Process diabetes dataset
    diabetes_df = data_processor.load_diabetes_data()
    X_train_diab, X_test_diab, y_train_diab, y_test_diab, diab_features = data_processor.preprocess_diabetes_data(diabetes_df)

    # Process skin segmentation dataset
    skin_df = data_processor.load_skin_data()
    X_train_skin, X_test_skin, y_train_skin, y_test_skin, skin_features = data_processor.preprocess_skin_data(skin_df)

    datasets = [
        ('diabetes', X_train_diab, X_test_diab, y_train_diab, y_test_diab),
        ('skin_segmentation', X_train_skin, X_test_skin, y_train_skin, y_test_skin)
    ]

    for dataset_name, X_train, X_test, y_train, y_test in datasets:
        print(f"\nAnalyzing {dataset_name} dataset...")

        # Linear SVM Analysis
        print(f"Training Linear SVM for {dataset_name}...")
        linear_analyzer = LinearSVMAnalyzer(run_folder, dataset_name)
        linear_results = linear_analyzer.train_model(X_train, X_test, y_train, y_test)
        linear_analyzer.plot_confusion_matrix(linear_results[2])
        linear_analyzer.save_results(*linear_results)

        # Non-linear SVM Analysis
        print(f"Training Non-linear SVMs for {dataset_name}...")
        nonlinear_analyzer = NonLinearSVMAnalyzer(run_folder, dataset_name)
        nonlinear_results = nonlinear_analyzer.train_models(X_train, X_test, y_train, y_test)

        # Cross-validation
        print(f"Performing cross-validation for {dataset_name}...")
        cv_results = nonlinear_analyzer.perform_cross_validation(X_train, y_train)

        # Hyperparameter tuning
        print(f"Tuning hyperparameters for {dataset_name}...")
        tuning_results = nonlinear_analyzer.tune_hyperparameters(X_train, y_train)

        # Save non-linear results
        nonlinear_analyzer.plot_kernel_comparison()
        nonlinear_analyzer.save_results(cv_results, tuning_results)

        # Overall comparison
        print(f"Creating comparison plots for {dataset_name}...")
        comparison = SVMComparison(run_folder, dataset_name)
        all_results = comparison.compare_all_models(linear_results, nonlinear_results)

        print(f"Results for {dataset_name}:")
        for model, acc in all_results.items():
            print(f"  {model}: {acc:.4f}")

    print(f"\nAnalysis complete! Results saved in folder: {run_folder}")

if __name__ == "__main__":
    main()

Loading and preprocessing datasets...
Loaded diabetes dataset: 768 samples, 8 features
Loaded skin segmentation dataset: 245057 samples, 3 features

Analyzing diabetes dataset...
Training Linear SVM for diabetes...
Training Non-linear SVMs for diabetes...
Performing cross-validation for diabetes...
Tuning hyperparameters for diabetes...
Creating comparison plots for diabetes...
Results for diabetes:
  Linear: 0.7208
  RBF: 0.7532
  Polynomial: 0.7532

Analyzing skin_segmentation dataset...
Training Linear SVM for skin_segmentation...
